In [ ]:
import os
if os.path.ismount('/content/drive'):
    print("Google Drive is mounted.")
else:
    print("Google Drive is not mounted.")

Google Drive is mounted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pymupdf python-docx tqdm

In [ ]:
!pip install gradio

In [ ]:
!pip install pytesseract pdf2image pymupdf Pillow python-docx

In [ ]:
# TEXT,PDF,DOCX WITH OCR
import os
import re
import spacy
import sqlite3
import docx
import fitz  # PyMuPDF for PDF processing
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from google.colab import files
from docx import Document
from tqdm import tqdm
import gradio as gr
from PIL import Image
from io import BytesIO

# Load NLP Model
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

# Function to extract text from PDF and save cleaned PDF
def extract_text_from_pdf(pdf_path, output_pdf_path):
    doc = fitz.open(pdf_path)
    new_doc = fitz.open()

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")  # Extract only text
        new_page = new_doc.new_page(width=page.rect.width, height=page.rect.height)
        new_page.insert_text((50, 50), text, fontsize=12)

    new_doc.save(output_pdf_path)
    new_doc.close()

# Function to extract text from DOCX and save cleaned DOCX
def extract_text_from_docx(docx_path, output_docx_path):
    doc = Document(docx_path)
    new_doc = Document()

    for para in doc.paragraphs:
        new_doc.add_paragraph(para.text)

    new_doc.save(output_docx_path)

# Ask the user if they want to upload a file or skip
upload_choice = input("Do you want to upload a file? (yes/no): ").strip().lower()

if upload_choice == "yes":
    print("Please upload your PDF or DOCX file:")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

# Check file format
file_ext = file_name.split(".")[-1].lower()

if file_ext == "pdf":
    output_file = "cleaned_output.pdf"
    extract_text_from_pdf(file_name, output_file)
elif file_ext == "docx":
    output_file = "cleaned_output.docx"
    extract_text_from_docx(file_name, output_file)
else:
    print("Unsupported file format. Please upload a PDF or DOCX file.")
    output_file = None

if output_file:
    print(f"Processing complete! Cleaned file saved as {output_file}")

# List of sections to exclude
EXCLUDED_SECTIONS = [
    "title page", "copyright", "dedication", "epigraph", "foreword", "preface", "acknowledgement", "introduction", "table of contents", "index",
    "prologue", "glossary", "list of figures", "letter from the author", "author's note", "cast of characters", "map", "timeline", "list of abbreviations"
]

def should_exclude(text):
    text_lower = text.lower()
    return any(section in text_lower for section in EXCLUDED_SECTIONS)

def remove_excluded_sections(doc):
    """Remove all excluded sections and their content from a DOCX document."""
    new_doc = docx.Document()
    skip = False

    for para in doc.paragraphs:
        text = para.text.strip().lower()

        if text in EXCLUDED_SECTIONS:
            skip = True  # Start skipping content after section title
            continue

        if skip and (not text or text.istitle() or text.isupper()):
            skip = False  # Stop skipping when a new section is reached

        if not skip:
            new_doc.add_paragraph(para.text.strip())

    return new_doc

# Function to clean text
def clean_text(text, header_footer_candidates):
    for candidate in header_footer_candidates:
        text = text.replace(candidate, '')
    text = re.sub(r'\n?\s*Page \d+\s*\n?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\n?\s*\d{1,2}/\d{1,2}/\d{2,4}\s*\n?', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Function to identify common headers/footers
def identify_header_footer(doc):
    line_freq = {}
    for page_num in range(len(doc)):
        page = doc[page_num]
        lines = page.get_text("text").split('\n')
        for line in lines:
            line = line.strip()
            if line:
                line_freq[line] = line_freq.get(line, 0) + 1
    threshold = len(doc) * 0.5
    return [line for line, freq in line_freq.items() if freq > threshold]

# Function to process PDF
def process_pdf(pdf_path):
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"Error opening PDF: {str(e)}")
        return []

    extracted_data = []
    header_footer_candidates = identify_header_footer(doc)

    for page_num in tqdm(range(len(doc)), desc="Processing PDF Pages"):
        try:
            page = doc[page_num]
            text = page.get_text("text").strip()

            if not text:
                images = convert_from_path(pdf_path, first_page=page_num + 1, last_page=page_num + 1)
                for image in images:
                    text = pytesseract.image_to_string(image).strip()

            if not text:
                continue

            if should_exclude(text):
                continue

            cleaned_text = clean_text(text, header_footer_candidates)
            extracted_data.append((page_num + 1, None, cleaned_text))
        except Exception as e:
            print(f"Error processing page {page_num}: {str(e)}")

    return extracted_data

# Function to process DOCX
def process_docx(docx_path):
    doc = docx.Document(docx_path)
    doc = remove_excluded_sections(doc)  # Remove unwanted sections like TOC, Preface etc.
    extracted_data = []
    paragraph_index = 1  # Initialize paragraph index

    for paragraph in tqdm(doc.paragraphs, desc="Processing DOCX Paragraphs"):
        text = paragraph.text.strip()

        # Skip empty paragraphs or ones with only whitespace
        if not text:
            continue

        # Skip paragraphs from excluded sections (like table of contents, etc.)
        if should_exclude(text):
            continue

        # Clean and store the valid paragraph
        cleaned_text = re.sub(r'\s+', ' ', text).strip()
        extracted_data.append((None, paragraph_index, cleaned_text))
        paragraph_index += 1  # Increment only for valid, non-empty paragraphs

    return extracted_data

# Store extracted data in SQLite
def store_in_database(data, db_name="documents.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS document_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            page_number INTEGER,
            paragraph_index INTEGER,
            text TEXT
        )
    """)
    cursor.executemany("INSERT INTO document_data (page_number, paragraph_index, text) VALUES (?, ?, ?)", data)
    conn.commit()
    conn.close()

def ensure_output_folder(folder="extracted_images"):
    if not os.path.exists(folder):
        os.makedirs(folder)
    return folder

def extract_images_from_pdf(pdf_path, output_folder):
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc[page_num]
        images = page.get_images(full=True)
        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            img_ext = base_image["ext"]
            image_filename = f"{output_folder}/pdf_page_{page_num+1}_img_{img_index+1}.{img_ext}"
            with open(image_filename, "wb") as img_file:
                img_file.write(image_bytes)
            store_image_metadata(image_filename, page_number=page_num+1)

def extract_images_from_docx(docx_path, output_folder):
    doc = docx.Document(docx_path)
    for para_idx, rel in enumerate(doc.part.rels):
        if "image" in doc.part.rels[rel].target_ref:
            image = doc.part.rels[rel].target_part.blob
            image_filename = f"{output_folder}/docx_para_{para_idx+1}.png"
            with open(image_filename, "wb") as img_file:
                img_file.write(image)
            store_image_metadata(image_filename, paragraph_index=para_idx+1)

def store_image_metadata(image_name, page_number=None, paragraph_index=None, db_name="images.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS image_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_name TEXT,
            page_number INTEGER,
            paragraph_index INTEGER
        )
    """)
    cursor.execute("INSERT INTO image_data (image_name, page_number, paragraph_index) VALUES (?, ?, ?)",
                   (image_name, page_number, paragraph_index))
    conn.commit()
    conn.close()

# Function to process uploaded file
def process_file(file):
    if file is None:
        return "No file uploaded."

    file_path = file.name
    output_folder = ensure_output_folder()

    extracted_data = []

    if file_path.lower().endswith(".pdf"):
        extracted_data = process_pdf(file_path)
        extract_images_from_pdf(file_path, output_folder)  # Extract images and store metadata
    elif file_path.lower().endswith(".docx"):
        extracted_data = process_docx(file_path)
        extract_images_from_docx(file_path, output_folder)  # Extract images and store metadata
    else:
        return "Unsupported file format! Only PDF and DOCX are supported."

    if extracted_data:
        store_in_database(extracted_data)  # Store extracted text in `documents.db`

    return "Processing complete! Extracted text stored in 'documents.db' and images stored in 'extracted_images' with metadata in 'images.db'."

# Process text input
def process_text_input(text):
    if not text.strip():
        return "No text entered."
    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    extracted_data = [(None, None, cleaned_text)]
    store_in_database(extracted_data)
    return "Processing complete! Data stored in 'documents.db'."

# Remove uploaded file
def remove_file():
    return None, "File removed successfully."

# Remove text from input
def remove_text():
    return "", "Text removed successfully."

# Gradio Interface
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="cyan"), css="""
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600&display=swap');

    .gradio-container {
        background-color: #e0f2ff; /* Light Blue */
        padding: 10px;
        font-family: 'Poppins', sans-serif;
    }
    .gr-box, .gr-panel {
        background-color: #f0f8ff; /* Alice Blue */
        border-radius: 12px;
        padding: 5px;
        box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
        max-height: 400px;
        overflow-y: auto;
        font-family: 'Poppins', sans-serif;
    }
    .gr-button {
        border-radius: 8px;
        background-color: #4682b4 !important; /* Steel Blue */
        color: white !important;
        font-family: 'Poppins', sans-serif;
    }
    .footer {
        display: none;
    }
""") as demo:
    gr.Markdown("# ✨ text2pix")
    with gr.Tab("Upload File"):
        file_input = gr.File(label="Upload PDF or DOCX File", file_types=[".pdf", ".docx"])
        process_button = gr.Button("Process File")
        remove_file_button = gr.Button("Remove File")

    with gr.Tab("Enter Text"):
        text_input = gr.Textbox(label="Enter text manually", lines=5)
        text_process_button = gr.Button("Process Text")
        remove_text_button = gr.Button("Remove Text")

    output_text = gr.Textbox(label="Status", interactive=False)

    process_button.click(fn=process_file, inputs=file_input, outputs=output_text)
    remove_file_button.click(fn=remove_file, inputs=[], outputs=[file_input, output_text])
    text_process_button.click(fn=process_text_input, inputs=text_input, outputs=output_text)
    remove_text_button.click(fn=remove_text, inputs=[], outputs=[text_input, output_text])

demo.launch(share=True)

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Do you want to upload a file? (yes/no): yes
Please upload your PDF or DOCX file:


Saving lekl101.pdf to lekl101.pdf
Processing complete! Cleaned file saved as cleaned_output.pdf
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a0a7aa333541b44b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install ipython-sql

In [ ]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
%sql sqlite:///documents.db

In [ ]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("documents.db")

# Read table data into a Pandas DataFrame
df = pd.read_sql_query("SELECT * FROM document_data;", conn)

# Display the data
print(df)

# Close the connection
conn.close()

   id  page_number paragraph_index  \
0   1            2            None   
1   2            3            None   
2   3            4            None   
3   4            5            None   
4   5            6            None   
5   6            8            None   
6   7            9            None   
7   8           10            None   
8   9           11            None   

                                                text  
0  2/KALEIDOSCOPE I S I S I S I S I Sell my Dream...  
1  3/I SELL MY DREAMS and everything returned to ...  
2  4/KALEIDOSCOPE paradise of black marketeering ...  
3  5/I SELL MY DREAMS covered her minor expenses,...  
4  6/KALEIDOSCOPE 1. How did the author recognise...  
5  8/KALEIDOSCOPE preparations that in some way r...  
6  9/I SELL MY DREAMS the temptation of questioni...  
7  10/KALEIDOSCOPE Language Work A. Vocabulary Lo...  
8  11/I SELL MY DREAMS TASK Study the following s...  


In [ ]:
import sqlite3

DB_NAME = "documents.db"

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("SELECT * FROM document_data")
rows = cursor.fetchall()

for row in rows:
    print(row)  # Prints all rows

conn.close()

(1, 2, None, '2/KALEIDOSCOPE I S I S I S I S I Sell my Dreams ell my Dreams ell my Dreams ell my Dreams ell my Dreams Gabriel Garcia Marquez was brought up by his grandparents in Northern Columbia because his parents were poor and struggling. A novelist, short- story writer and journalist, he is widely considered the greatest living Latin American master of narrative. Marquez won the Nobel Prize in Literature in 1982. His two masterpieces are One Hundred Years in Solitude (1967, tr. 1970) and Love in The Time of Cholera (1985, tr. 1988). His themes are violence, solitude and the overwhelming human need for love. This story reflects, like most of his works, a high point in Latin American magical realism; it is rich and lucid, mixing reality with fantasy. One morning at nine o’clock, while we were having breakfast on the terrace of the Havana Riviera Hotel under a bright sun, a huge wave picked up several cars that were driving down the avenue along the seawall or parked on the pavement,

In [ ]:
# DELETION
import sqlite3
import pandas as pd

conn = sqlite3.connect("documents.db")
cursor = conn.cursor()

cursor.execute("DELETE FROM document_data;")
cursor.execute("DELETE FROM sqlite_sequence WHERE name='document_data';")

conn.commit()

df = pd.read_sql_query("SELECT * FROM document_data;", conn)
print(df)

conn.close()

OperationalError: database is locked

In [ ]:
# ROW DELETION IN TABLE
import sqlite3
import pandas as pd

conn = sqlite3.connect("documents.db")
cursor = conn.cursor()

# Delete rows where paragraph_index is 1, 3, or 5
cursor.execute("DELETE FROM document_data WHERE paragraph_index in (1,2,4,5,7,8,10,11,13,14,15,17,18,20,21,22,24,25,26,28,29,30,32,33,34,36);")

conn.commit()

# Verify by reading the remaining data
df = pd.read_sql_query("SELECT * FROM document_data;", conn)
print(df)

conn.close()

OperationalError: database is locked

In [ ]:
# CHANGING PARAGRAPH_INDEX
import sqlite3
import pandas as pd

conn = sqlite3.connect("documents.db")
cursor = conn.cursor()

# New values for id and paragraph_index
update_mapping = {
    3: 1,
    6: 2,
    9: 3,
    12: 4,
    16: 5,
    19: 6,
    23: 7,
    27: 8,
    31: 9,
    35: 10
}

# Update both id and paragraph_index
for old_id, new_val in update_mapping.items():
    cursor.execute("""
        UPDATE document_data
        SET id = ?, paragraph_index = ?
        WHERE id = ?;
    """, (new_val, new_val, old_id))

conn.commit()

# Verify update
df = pd.read_sql_query("SELECT * FROM document_data WHERE id IN (1,2,3,4,5,6,7,8,9,10);", conn)
print(df)

conn.close()

IntegrityError: UNIQUE constraint failed: document_data.id

In [ ]:
!pip install huggingface_hub

In [ ]:
!pip install gensim
!pip install --upgrade pip setuptools wheel
!pip uninstall -y scipy
!pip install --no-cache-dir scipy
!pip install --no-cache-dir gensim scikit-learn numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 17.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 316.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 225.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.2
    Uninstalling scipy-1.15.2:
      Successfully uninstalled scipy-1.15.2


In [ ]:
# MOUNTED CODE(GENERATES SINGLE IMAGE PER PARA)
import sqlite3
import re
import time
import spacy  # NLP for subject-verb extraction
from math import ceil  # For dynamic threshold
from huggingface_hub import InferenceClient
from io import BytesIO
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api
import sys
import os

# Load English NLP model
nlp = spacy.load("en_core_web_sm")

# Add Google Drive library path
sys.path.append('/content/drive/MyDrive/colab_libs')

# Define path to save/load the Word2Vec model
model_path = '/content/drive/MyDrive/word2vec-google-news-300/word2vec-google-news-300.model'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Check if model exists in Drive, else upload/download
if os.path.exists(model_path):
    print("📁 Loading Word2Vec model from Google Drive...")
    model = KeyedVectors.load(model_path)
    print("✅ Model loaded successfully.")
else:
    print("⬇️ Downloading Word2Vec model...")
    model = api.load("word2vec-google-news-300")
    model.save(model_path)
    print("✅ Model saved to Google Drive.")

# Hugging Face API Key (Replace with your key)
API_KEY = "your_api_key"
MODEL = "stabilityai/stable-diffusion-3.5-large-turbo"

client = InferenceClient(model=MODEL, token=API_KEY)

# SQLite Database
DB_NAME = "documents.db"

def fetch_paragraphs_without_images():
    """Fetch paragraphs where has_image = 0."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text FROM document_data")
    paragraphs = cursor.fetchall()
    conn.close()
    return paragraphs

def extract_context(sentence):
    """Tokenize and extract words from a sentence (simple example)."""
    return sentence.split()

def icf_algorithm(sentences):
    """Apply Initial Context First (ICF) ensuring correct subject-verb context."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n):
        S_cxt[i] += W_cxt[0]
        S_input[i] = " ".join(S_cxt[i])

    return " ".join(S_input)

def rcf_algorithm(sentences, delta):
    """Apply Region Context First (RCF) with a dynamic threshold."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n):
        if n > delta:
            S_cxt[i] += W_cxt[i % delta]
        else:
            S_cxt[i] += W_cxt[0]
        S_input[i] = " ".join(S_cxt[i])

    return " ".join(S_input)

def calculate_similarity(context1, context2, model):
    """Calculate cosine similarity between two contexts using Word2Vec embeddings."""
    vector1 = sentence_to_vector(context1, model)
    vector2 = sentence_to_vector(context2, model)

    if vector1 is not None and vector2 is not None:
        return cosine_similarity([vector1], [vector2])[0][0]
    return 0

def sentence_to_vector(sentence, model):
    """Convert a sentence into a vector by averaging Word2Vec embeddings."""
    vectors = [model[word] for word in sentence if word in model.key_to_index]
    return np.mean(vectors, axis=0) if vectors else None

def scrm_algorithm(sentences, delta):
    """Apply Semantic Context Recognition and Modification (SCRM)."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []
    cxtIndex = 1

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n - 1):
        similarity = calculate_similarity(W_cxt[i], W_cxt[i + 1], model)

        if similarity > delta:
            S_cxt[i] += W_cxt[cxtIndex - 1]
        else:
            cxtIndex = i + 1
            S_cxt[i] += W_cxt[cxtIndex - 1]

        S_input[i] = " ".join(S_cxt[i])

    return " ".join(S_input)

def generate_image_with_retry(prompt, output_filename, max_retries=5):
    """Generate an image with retries and delay handling."""
    retries = 0
    while retries < max_retries:
        try:
            image = client.text_to_image(prompt, width=512, height=512)
            image_bytes = BytesIO()
            image.save(image_bytes, format="PNG")
            with open(output_filename, "wb") as f:
                f.write(image_bytes.getvalue())
            print(f"✅ Image saved as {output_filename}")
            return
        except Exception as e:
            print(f"⚠️ Error: {e}. Retrying in {10 * (retries + 1)} seconds...")
            time.sleep(10 * (retries + 1))
            retries += 1
    print("❌ Max retries reached. Skipping this request.")

def update_has_image_status(paragraph_id):
    """Mark the paragraph as processed in the database."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    # cursor.execute("UPDATE document_data SET has_image = 1 WHERE id = ?", (paragraph_id,))
    conn.commit()
    conn.close()
    print(f"🔄 Updated has_image = TRUE for paragraph ID {paragraph_id}")

# Fetch paragraphs without images
paragraphs = fetch_paragraphs_without_images()

# Process each paragraph
for paragraph_id, text in paragraphs:
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    if len(sentences) <= 5:
        prompt = icf_algorithm(sentences)
    elif len(sentences) <= 10:
        prompt = rcf_algorithm(sentences, delta=max(3, ceil(len(sentences) / 2)))
    else:
        prompt = scrm_algorithm(sentences, delta=0.7)  # Adjust similarity threshold as needed

    generate_image_with_retry(prompt, f"output_{paragraph_id}.png")
    time.sleep(20)
    update_has_image_status(paragraph_id)

print("🎉 All paragraphs processed successfully!")

ImportError: scipy.special._ufuncs_cxx does not export expected C variable _export_ccospi

In [ ]:
# CHATBOT
import sqlite3
import re
import time
import spacy
from math import ceil
from huggingface_hub import InferenceClient
from PIL import Image  # For displaying images
from io import BytesIO
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api
import sys
import os

# Load English NLP model
nlp = spacy.load("en_core_web_sm")

# Add Google Drive library path
sys.path.append('/content/drive/MyDrive/colab_libs')

# Define path to save/load the Word2Vec model
model_path = '/content/drive/MyDrive/word2vec-google-news-300/word2vec-google-news-300.model'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Check if model exists in Drive, else upload/download
if os.path.exists(model_path):
    print("Loading Word2Vec model from Google Drive...")
    model = KeyedVectors.load(model_path)
    print("Model loaded successfully.")
else:
    print("Downloading Word2Vec model...")
    model = api.load("word2vec-google-news-300")
    model.save(model_path)
    print("Model saved to Google Drive.")

# Hugging Face API Key (Replace with your key)
API_KEY = "your_api_key"
MODEL = "stabilityai/stable-diffusion-3.5-large"

client = InferenceClient(model=MODEL, token=API_KEY)

# SQLite Database
DB_NAME = "documents.db"

def fetch_paragraphs_without_images():
    """Fetch paragraphs where has_image = 0."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text FROM document_data")
    paragraphs = cursor.fetchall()
    conn.close()
    return paragraphs

def extract_context(sentence):
    """Tokenize and extract words from a sentence (simple example)."""
    return sentence.split()

def icf_algorithm(sentences):
    """Apply Initial Context First (ICF) ensuring correct subject-verb context."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n):
        S_cxt[i] += W_cxt[0]
        S_input[i] = " ".join(S_cxt[i])

    return " ".join(S_input)

def rcf_algorithm(sentences, delta):
    """Apply Region Context First (RCF) with a dynamic threshold."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n):
        if n > delta:
            S_cxt[i] += W_cxt[i % delta]
        else:
            S_cxt[i] += W_cxt[0]
        S_input[i] = " ".join(S_cxt[i])

    return " ".join(S_input)

def calculate_similarity(context1, context2, model):
    """Calculate cosine similarity between two contexts using Word2Vec embeddings."""
    vector1 = sentence_to_vector(context1, model)
    vector2 = sentence_to_vector(context2, model)

    if vector1 is not None and vector2 is not None:
        return cosine_similarity([vector1], [vector2])[0][0]
    return 0

def sentence_to_vector(sentence, model):
    """Convert a sentence into a vector by averaging Word2Vec embeddings."""
    vectors = [model[word] for word in sentence if word in model.key_to_index]
    return np.mean(vectors, axis=0) if vectors else None

def scrm_algorithm(sentences, delta):
    """Apply Semantic Context Recognition and Modification (SCRM)."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []
    cxtIndex = 1

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n - 1):
        similarity = calculate_similarity(W_cxt[i], W_cxt[i + 1], model)

        if similarity > delta:
            S_cxt[i] += W_cxt[cxtIndex - 1]
        else:
            cxtIndex = i + 1
            S_cxt[i] += W_cxt[cxtIndex - 1]

        S_input[i] = " ".join(S_cxt[i])

    return " ".join(S_input)

def generate_image_with_retry(prompt, output_filename, max_retries=5):
    """Generate an image with retries and return the output filename."""
    retries = 0
    while retries < max_retries:
        try:
            image = client.text_to_image(prompt, width=512, height=512)
            image_bytes = BytesIO()
            image.save(image_bytes, format="PNG")
            with open(output_filename, "wb") as f:
                f.write(image_bytes.getvalue())
            print(f"Image saved as {output_filename}")
            return output_filename  # ✅ RETURN FILENAME
        except Exception as e:
            print(f"Error: {e}. Retrying in {10 * (retries + 1)} seconds...")
            time.sleep(10 * (retries + 1))
            retries += 1
    print("Max retries reached. Skipping this request.")
    return None  # ✅ RETURN None if failed
"""
def update_has_image_status(paragraph_id):
    Mark the paragraph as processed in the database
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("UPDATE document_data SET has_image = 1 WHERE id = ?", (paragraph_id,))
    conn.commit()
    conn.close()
    print(f"Updated has_image = TRUE for paragraph ID {paragraph_id}")
"""
def get_user_feedback(image_path):
    """Ask the user if they are satisfied with the generated image."""
    image = Image.open(image_path)
    image.show()  # Display the image

    while True:
        feedback = input("Are you satisfied with this image? (yes/no): ").strip().lower()
        if feedback in ["yes", "no"]:
            return feedback == "yes"  # Return True if yes, False if no
        print("Invalid input. Please enter 'yes' or 'no'.")

# Fetch paragraphs without images
paragraphs = fetch_paragraphs_without_images()

# Process each paragraph
import random  # ✅ Import random for variation

for paragraph_id, text in paragraphs:
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    delta = max(3, ceil(len(sentences) / 2))

    # Choose the prompt editing method
    if len(sentences) <= 5:
        base_prompt = icf_algorithm(sentences)
    elif len(sentences) <= 10:
        base_prompt = rcf_algorithm(sentences, delta=max(3, ceil(len(sentences) / 2)))
    else:
        base_prompt = scrm_algorithm(sentences, delta=0.7)  # Adjust similarity threshold as needed

    attempt = 1  # ✅ Track attempts
    while True:
        # Modify prompt slightly to generate a different image each time
        varied_prompt = f"{base_prompt} | variation {random.randint(1000, 9999)}"  # ✅ Adds randomness

        output_filename = f"output_{paragraph_id}_attempt{attempt}.png"
        image_path = generate_image_with_retry(varied_prompt, output_filename)  # ✅ Pass varied prompt

        if image_path:  # ✅ Ensure an image is generated
            if get_user_feedback(image_path):  # ✅ Ask for feedback
                # update_has_image_status(paragraph_id)
                break  # ✅ Stop if satisfied
            else:
                print("Generating a new image with slight prompt variation...")  # ✅ Notify user
                attempt += 1  # ✅ Increment attempt count

print("All paragraphs processed successfully!")

Loading Word2Vec model from Google Drive...
Model loaded successfully.
Image saved as output_1_attempt1.png
Are you satisfied with this image? (yes/no): no
Generating a new image with slight prompt variation...
Image saved as output_1_attempt2.png
Are you satisfied with this image? (yes/no): yes
Image saved as output_2_attempt1.png
Are you satisfied with this image? (yes/no): yes
Image saved as output_3_attempt1.png
Are you satisfied with this image? (yes/no): yes
All paragraphs processed successfully!


In [ ]:
# INSERTION
import os
import sqlite3
import docx
from docx.shared import Inches
from tqdm import tqdm

DB_NAME = "documents.db"

def fetch_paragraphs_without_images():
    """Fetch paragraphs that need images from the database in order."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text FROM document_data ORDER BY id")
    paragraphs = cursor.fetchall()
    conn.close()
    return paragraphs  # Returns a list of (para_id, text)

def get_latest_image(para_id):
    """Fetch the latest generated image for a paragraph."""
    attempt = 1
    latest_image = None

    while True:
        image_path = f"output_{para_id}_attempt{attempt}.png"
        if os.path.exists(image_path):
            latest_image = image_path
            attempt += 1
        else:
            break
    return latest_image  # Returns the last valid image

def insert_images_into_docx(original_docx, output_docx):
    """Insert images into a DOCX document right after their respective paragraphs."""
    print(f"Processing DOCX: {original_docx}")
    doc = docx.Document(original_docx)
    paragraphs = fetch_paragraphs_without_images()

    for para_id, paragraph_text in tqdm(paragraphs, desc="Inserting images into DOCX"):
        image_path = get_latest_image(para_id)
        if not image_path:
            print(f"Warning: No image found for paragraph {para_id}. Skipping...")
            continue

        # Search for the exact paragraph in the document
        for i, para in enumerate(doc.paragraphs):
            if para.text.strip() == paragraph_text.strip():
                # Insert a new paragraph immediately after the matched paragraph
                new_para = doc.add_paragraph()  # Create a new paragraph
                new_run = new_para.add_run()  # Add a run to the new paragraph
                new_run.add_picture(image_path, width=Inches(3))  # Medium-sized image
 # Insert image

                # Move new paragraph to be right after the current paragraph
                doc._element.body.insert(i + 1, new_para._element)

                print(f"Inserted image {image_path} below paragraph {para_id}")
                break  # Move to the next paragraph in the database

    doc.save(output_docx)
    print(f"🎉 Updated DOCX saved as: {output_docx}")

def process_and_update_file(original_file):
    """Determine file type and process accordingly."""
    if original_file.lower().endswith(".docx"):
        output_file = "updated_document.docx"
        insert_images_into_docx(original_file, output_file)
    else:
        print("Unsupported file format!")
        return None
    return output_file

# File processing
original_file = "monuments.docx"
updated_file = process_and_update_file(original_file)

if updated_file:
    print(f"Download your updated file: {updated_file}")

Processing DOCX: monuments.docx


Inserting images into DOCX: 100%|██████████| 10/10 [00:00<00:00, 105.37it/s]

Inserted image output_10_attempt1.png below paragraph 10


🎉 Updated DOCX saved as: updated_document.docx
Download your updated file: updated_document.docx


In [ ]:
# INSERTION WITH CITATION
import os
import sqlite3
import docx
from docx.shared import Inches
from tqdm import tqdm

DB_NAME = "documents.db"

def fetch_paragraphs_without_images():
    """Fetch paragraphs that need images from the database in order."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text FROM document_data ORDER BY id")
    paragraphs = cursor.fetchall()
    conn.close()
    return paragraphs  # Returns a list of (para_id, text)

def get_latest_image(para_id):
    """Fetch the latest generated image for a paragraph."""
    attempt = 1
    latest_image = None

    while True:
        image_path = f"output_{para_id}_attempt{attempt}.png"
        if os.path.exists(image_path):
            latest_image = image_path
            attempt += 1
        else:
            break
    return latest_image  # Returns the last valid image

def ensure_caption_style(doc):
    """Ensure the 'Caption' style exists, or create it if missing."""
    styles = doc.styles
    if "Caption" not in styles:
        caption_style = styles.add_style("Caption", docx.enum.style.WD_STYLE_TYPE.PARAGRAPH)
        caption_style.font.bold = True  # Make captions bold (optional)

def insert_images_into_docx(original_docx, output_docx):
    """Insert images into a DOCX document right after their respective paragraphs, with captions."""
    print(f"Processing DOCX: {original_docx}")
    doc = docx.Document(original_docx)
    paragraphs = fetch_paragraphs_without_images()

    ensure_caption_style(doc)  # Ensure Caption style exists

    figure_count = 1  # Counter for figure numbers

    for para_id, paragraph_text in tqdm(paragraphs, desc="Inserting images into DOCX"):
        image_path = get_latest_image(para_id)
        if not image_path:
            print(f"Warning: No image found for paragraph {para_id}. Skipping...")
            continue

        # Search for the exact paragraph in the document
        for i, para in enumerate(doc.paragraphs):
            if para.text.strip() == paragraph_text.strip():
                # Insert a new paragraph immediately after the matched paragraph
                new_para = doc.add_paragraph()
                new_run = new_para.add_run()
                new_run.add_picture(image_path, width=Inches(3))  # Insert image

                # Move new paragraph to be right after the current paragraph
                doc._element.body.insert(i + 1, new_para._element)

                # Add a caption below the image
                caption_para = doc.add_paragraph(f"Figure {figure_count}: Image for paragraph {para_id}")
                caption_para.style = "Caption"  # Apply caption style
                doc._element.body.insert(i + 2, caption_para._element)

                print(f"Inserted image {image_path} with caption below paragraph {para_id}")
                figure_count += 1  # Increment figure counter
                break  # Move to the next paragraph in the database

    doc.save(output_docx)
    print(f"🎉 Updated DOCX saved as: {output_docx}")

def process_and_update_file(original_file):
    """Determine file type and process accordingly."""
    if original_file.lower().endswith(".docx"):
        output_file = "updated_document.docx"
        insert_images_into_docx(original_file, output_file)
    else:
        print("Unsupported file format!")
        return None
    return output_file

# File processing
original_file = "cleaned_output.docx"
updated_file = process_and_update_file(original_file)

if updated_file:
    print(f"Download your updated file: {updated_file}")

Processing DOCX: cleaned_output.docx


Inserting images into DOCX: 100%|██████████| 3/3 [00:00<00:00, 246.31it/s]

Inserted image output_1_attempt2.png with caption below paragraph 1
Inserted image output_2_attempt1.png with caption below paragraph 2
Inserted image output_3_attempt1.png with caption below paragraph 3
🎉 Updated DOCX saved as: updated_document.docx
Download your updated file: updated_document.docx


In [ ]:
!pip install torch torchvision transformers pillow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 164.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nv

In [ ]:
# COMPARISON OF ORIGINAL AND GENERATED IMAGES
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

# Load CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Smart truncation for long paragraphs (CLIP max is ~77 tokens)
def smart_truncate(text, max_words=75):
    words = text.split()
    if len(words) > max_words:
        return " ".join(words[:max_words])
    return text

# Function to get image embedding
def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)
    return image_features.cpu().numpy()

# Function to get text embedding
def get_text_embedding(text):
    text = smart_truncate(text)
    inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    return text_features.cpu().numpy()

# === SET THESE VARIABLES ===
paragraph = """
Christmas - The Season of Joy-Christmas is a Christian festival celebrating the birth of Jesus Christ on December 25th. It is marked by decorating Christmas trees, exchanging gifts, singing carols, and attending church services. Families gather for feasts, and children eagerly await Santa Claus and his presents.
"""

original_image_path = "extracted_images/docx_para_6.png"  # extracted_images/pdf_page_11_img_102.png(path to original extracted image)
generated_image_path = "output_1_attempt1.png"  # path to generated image

# Get embeddings
paragraph_embedding = get_text_embedding(paragraph)
original_embedding = get_image_embedding(original_image_path)
generated_embedding = get_image_embedding(generated_image_path)

# Compute similarities
original_score = cosine_similarity(original_embedding, paragraph_embedding)[0][0]
generated_score = cosine_similarity(generated_embedding, paragraph_embedding)[0][0]

# Display results
print("\n====== COMPARISON RESULT ======")
print(f"Original image similarity:  {original_score:.4f}")
print(f"Generated image similarity: {generated_score:.4f}")

if generated_score > original_score:
    print("✅ The generated image is more relevant to the paragraph.")
else:
    print("🖼️ The original extracted image is more relevant to the paragraph.")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]


====== COMPARISON RESULT ======
Original image similarity:  0.2636
Generated image similarity: 0.2456
🖼️ The original extracted image is more relevant to the paragraph.


In [ ]:
#RANKING
import sqlite3
import os
import torch
from PIL import Image
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm
from torch.nn.functional import cosine_similarity

# Load CLIP model and processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Connect to SQLite DB
conn = sqlite3.connect("documents.db")
cursor = conn.cursor()

# Get all paragraphs (you can filter with WHERE has_image = 0 if needed)
cursor.execute("SELECT id, paragraph_index, text FROM document_data")
paragraphs = cursor.fetchall()

# Ranking function for each paragraph using cosine similarity
def rank_images_for_paragraph(paragraph_id, para_index, paragraph_text):
    image_scores = []

    # Match image files like output_1_attempt*.png
    image_files = [
        f for f in os.listdir(".")
        if f.startswith(f"output_{para_index}_attempt") and f.lower().endswith((".jpg", ".png", ".jpeg"))
    ]

    if not image_files:
        print(f"[!] No generated images for paragraph {para_index}")
        return []

    for image_file in image_files:
        img_path = os.path.join(".", image_file)

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[x] Failed to load image {img_path}: {e}")
            continue

        # Encode with CLIP
        inputs = clip_processor(text=paragraph_text, images=image, return_tensors="pt", padding=True, truncation=True)

        with torch.no_grad():
            outputs = clip_model(**inputs)

            # Get cosine similarity between image and text embeddings
            image_features = outputs.image_embeds
            text_features = outputs.text_embeds
            score = cosine_similarity(image_features, text_features).item()

        image_scores.append((img_path, score))

    # Sort images by score descending
    image_scores.sort(key=lambda x: x[1], reverse=True)

    return image_scores

# Run ranking on all paragraphs
all_rankings = {}  # Dictionary to store ranked images for each paragraph
for pid, pidx, ptext in tqdm(paragraphs, desc="Ranking Images"):
    ranked = rank_images_for_paragraph(pid, pidx, ptext)
    if ranked:
        all_rankings[(pid, pidx)] = ranked

# Display results with best image highlighted
print("\n🔍 Final Ranked Results Per Paragraph:\n")
for (pid, pidx), ranked_images in all_rankings.items():
    print(f"Paragraph ID: {pid}, Paragraph Index: {pidx}")
    for i, (img_path, score) in enumerate(ranked_images, 1):
        print(f"  {i}. {os.path.basename(img_path)} - Score: {score:.4f}")
    if ranked_images:
        best_img_path, best_score = ranked_images[0]
        print(f"✅ Best Image: {os.path.basename(best_img_path)} (Score: {best_score:.4f})")
    print("-" * 50)

Ranking Images: 100%|██████████| 2/2 [00:00<00:00,  3.43it/s]


🔍 Final Ranked Results Per Paragraph:

Paragraph ID: 1, Paragraph Index: 1
  1. output_1_attempt1.png - Score: 0.2456
✅ Best Image: output_1_attempt1.png (Score: 0.2456)
--------------------------------------------------
Paragraph ID: 2, Paragraph Index: 2
  1. output_2_attempt1.png - Score: 0.2878
✅ Best Image: output_2_attempt1.png (Score: 0.2878)
--------------------------------------------------


In [ ]:
# MOUNTED CODE (GENERATES MULTIPLE IMAGE PER PARA)
import sqlite3
import re
import time
import spacy  # NLP for subject-verb extraction
from math import ceil  # For dynamic threshold
from huggingface_hub import InferenceClient
from io import BytesIO
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api
import sys
import os

# Load English NLP model
nlp = spacy.load("en_core_web_sm")

# Add Google Drive library path
sys.path.append('/content/drive/MyDrive/colab_libs')

# Define path to save/load the Word2Vec model
model_path = '/content/drive/MyDrive/word2vec-google-news-300/word2vec-google-news-300.model'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Check if model exists in Drive, else upload/download
if os.path.exists(model_path):
    print("📁 Loading Word2Vec model from Google Drive...")
    model = KeyedVectors.load(model_path)
    print("✅ Model loaded successfully.")
else:
    print("⬇️ Downloading Word2Vec model...")
    model = api.load("word2vec-google-news-300")
    model.save(model_path)
    print("✅ Model saved to Google Drive.")

# Hugging Face API Key (Replace with your key)
API_KEY = "your_api_key"
MODEL = "stabilityai/stable-diffusion-3.5-large"

client = InferenceClient(model=MODEL, token=API_KEY)

# SQLite Database
DB_NAME = "documents.db"

def fetch_paragraphs_without_images():
    """Fetch paragraphs where has_image = 0."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text FROM document_data WHERE has_image = 0")
    paragraphs = cursor.fetchall()
    conn.close()
    return paragraphs

def extract_context(sentence):
    """Tokenize and extract words from a sentence (simple example)."""
    return sentence.split()

def icf_algorithm(sentences):
    """Apply Initial Context First (ICF) ensuring correct subject-verb context."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n):
        S_cxt[i] += W_cxt[0]
        S_input[i] = " ".join(S_cxt[i])

    return [" ".join(S_cxt[i]) for i in range(n)]

def rcf_algorithm(sentences, delta):
    """Apply Region Context First (RCF) with a dynamic threshold."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n):
        if n > delta:
            S_cxt[i] += W_cxt[i % delta]
        else:
            S_cxt[i] += W_cxt[0]
        S_input[i] = " ".join(S_cxt[i])

    return [" ".join(S_cxt[i]) for i in range(n)]

def calculate_similarity(context1, context2, model):
    """Calculate cosine similarity between two contexts using Word2Vec embeddings."""
    vector1 = sentence_to_vector(context1, model)
    vector2 = sentence_to_vector(context2, model)

    if vector1 is not None and vector2 is not None:
        return cosine_similarity([vector1], [vector2])[0][0]
    return 0

def sentence_to_vector(sentence, model):
    """Convert a sentence into a vector by averaging Word2Vec embeddings."""
    vectors = [model[word] for word in sentence if word in model.key_to_index]
    return np.mean(vectors, axis=0) if vectors else None

def scrm_algorithm(sentences, delta):
    """Apply Semantic Context Recognition and Modification (SCRM)."""
    n = len(sentences)
    S_input = sentences[:]
    S_cxt = []
    W_cxt = []
    cxtIndex = 1

    for i in range(n):
        sentence_context = extract_context(sentences[i])
        S_cxt.append(sentence_context)
        W_cxt.append(sentence_context)

    for i in range(n - 1):
        similarity = calculate_similarity(W_cxt[i], W_cxt[i + 1], model)

        if similarity > delta:
            S_cxt[i] += W_cxt[cxtIndex - 1]
        else:
            cxtIndex = i + 1
            S_cxt[i] += W_cxt[cxtIndex - 1]

        S_input[i] = " ".join(S_cxt[i])

    return [" ".join(S_cxt[i]) for i in range(n)]

def generate_image_with_retry(prompt, output_filename, max_retries=5):
    """Generate an image with retries and delay handling."""
    retries = 0
    while retries < max_retries:
        try:
            image = client.text_to_image(prompt, width=512, height=512)
            image_bytes = BytesIO()
            image.save(image_bytes, format="PNG")
            with open(output_filename, "wb") as f:
                f.write(image_bytes.getvalue())
            print(f"✅ Image saved as {output_filename}")
            return
        except Exception as e:
            print(f"⚠️ Error: {e}. Retrying in {10 * (retries + 1)} seconds...")
            time.sleep(10 * (retries + 1))
            retries += 1
    print("❌ Max retries reached. Skipping this request.")

def update_has_image_status(paragraph_id):
    """Mark the paragraph as processed in the database."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("UPDATE document_data SET has_image = 1 WHERE id = ?", (paragraph_id,))
    conn.commit()
    conn.close()
    print(f"🔄 Updated has_image = TRUE for paragraph ID {paragraph_id}")

# Fetch paragraphs without images
paragraphs = fetch_paragraphs_without_images()

# Process each paragraph
for paragraph_id, text in paragraphs:
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    if len(sentences) <= 5:
        processed_texts = icf_algorithm(sentences)
    elif len(sentences) <= 10:
        processed_texts = rcf_algorithm(sentences, delta=max(3, ceil(len(sentences) / 2)))
    else:
        processed_texts = scrm_algorithm(sentences, delta=0.7)

    for i, prompt in enumerate(processed_texts):
        generate_image_with_retry(prompt, f"output_{paragraph_id}_{i+1}.png")
        time.sleep(20)
    update_has_image_status(paragraph_id)

print("🎉 All paragraphs processed successfully!")

📁 Loading Word2Vec model from Google Drive...


In [ ]:
! pip install pillow

In [ ]:
# COMBINING IMAGES INTO STRIPS
import glob
from PIL import Image

def combine_images_horizontally_dynamic(input_pattern, output_path):
    # Dynamically find all images matching the pattern
    image_paths = sorted(glob.glob(input_pattern))

    if not image_paths:
        print("No images found with the given pattern.")
        return

    # Load images
    images = [Image.open(img) for img in image_paths]

    # Ensure all images have the same height
    heights = [img.height for img in images]
    max_height = max(heights)
    resized_images = [img.resize((int(img.width * max_height / img.height), max_height)) for img in images]

    # Calculate total width
    total_width = sum(img.width for img in resized_images)

    # Create a blank canvas
    combined_image = Image.new('RGB', (total_width, max_height))

    # Paste images side by side
    x_offset = 0
    for img in resized_images:
        combined_image.paste(img, (x_offset, 0))
        x_offset += img.width

    # Save the combined image
    combined_image.save(output_path)
    print(f"Combined image saved at {output_path}")

# Example usage:
# This will match files like output_1.png, output_2.png, etc.
combine_images_horizontally_dynamic("output_*.png", "combined_strip.png")

Evaluation

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

# Load BLIP-2 model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to("cuda" if torch.cuda.is_available() else "cpu")

def generate_paragraph_caption(image_path):
    image = Image.open(image_path).convert("RGB")

    # Use a detailed prompt
    prompt = "Describe this image in a detailed paragraph."

    # Process inputs with text prompt
    inputs = processor(image, text=prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    # Generate output
    with torch.no_grad():
        caption_ids = model.generate(
            **inputs,
            max_length=150,  # Increase for longer text
            num_beams=5,     # Beam search for better quality
            no_repeat_ngram_size=2,
            repetition_penalty=1.5,
            early_stopping=True
        )

    caption = processor.decode(caption_ids[0], skip_special_tokens=True)
    return caption

# Example usage
image_path = "output_1_attempt1.png"  # Replace with your image path
paragraph_caption = generate_paragraph_caption(image_path)
print(f"\nGenerated Paragraph Caption:\n{paragraph_caption}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]


Generated Paragraph Caption:
describe this image in a detailed paragraph. of the human head and neck
